# Prompt caching in Microsoft Foundry — a runnable demonstration

This notebook makes each mechanism in the deck observable against a real deployment.
Every demonstration prints `cached_tokens` so you can watch the cache hit, miss, and recover.

| Demo | What it shows | Slide |
|---|---|---|
| 1 | Cold call writes the prefix; the warm call reads it | How it works |
| 2 | One changed character early gives `cached_tokens = 0` | How hits are lost |
| 3 | Static-first beats dynamic-first prompt layout | The rule strip |
| 4 | Append-only multi-turn growth in reuse | How hits are lost |
| 5 | Explicit breakpoints and `prompt_cache_key` | The choices that matter |
| 6 | Read/write ratio as an operational signal | Operations and boundaries |

**Facts in this notebook come from the Microsoft Learn page**
[Prompt caching with Azure OpenAI in Microsoft Foundry Models](https://learn.microsoft.com/en-us/azure/foundry/openai/how-to/prompt-caching).

---

## Before you run anything

This notebook uses the same project, `.env` and packages as notebooks 1 and 2:

1. `FOUNDRY_PROJECT_ENDPOINT` and `FOUNDRY_MODEL_NAME` in the root `.env`. Calls go through the project's
   OpenAI-compatible client (`project.get_openai_client()`) and the **Responses API**.
   **Demo 5 requires a GPT-5.6-or-later model on a Standard pay-as-you-go deployment** —
   earlier models return a 400 error if the request includes `prompt_cache_options`
   or `prompt_cache_breakpoint`, and Provisioned Throughput managed (PTU-M)
   deployments do not support breakpoints.
2. The **Foundry User** role on the project, for keyless authentication with `DefaultAzureCredential`.
3. `pip install -r requirements.txt` (includes `tiktoken` for the offline prefix-length check).

> **Caching is not deterministic.** Cached states live on individual machines, and a request
> reuses a prefix only if it reaches a machine still holding a matching, unexpired entry.
> A demo can legitimately report a miss where you expected a hit. Re-run it before concluding
> anything — and never build a cost model on a single observation.


In [1]:
import os
import time

from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

load_dotenv()
PROJECT_ENDPOINT = os.getenv("FOUNDRY_PROJECT_ENDPOINT", "https://<your-foundry-account>.services.ai.azure.com/api/projects/<your-project>")
DEPLOYMENT = os.getenv("FOUNDRY_MODEL_NAME", "<chat-model-deployment-name>")

credential = DefaultAzureCredential()
project = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)
openai_client = project.get_openai_client()

print(f"project    : {PROJECT_ENDPOINT}")
print(f"deployment : {DEPLOYMENT}")


project    : https://cog-tb7tpjtuee4ji.services.ai.azure.com/api/projects/cog-tb7tpjtuee4ji-project
deployment : gpt-6-astra


## The measurement helper

Everything below reports the same three numbers from the Responses API `usage` object:

- **`input_tokens`** — how much input the request carried.
- **`cached_tokens`** — the cache *read*, found under `input_tokens_details`.
  A cache hit is a match between a prompt's token computations and the current cache content.
- **`cache_write_tokens`** — the cache *write*, also under `input_tokens_details`. Reported on Standard
  pay-as-you-go deployments with GPT-5.6 and later models; absent elsewhere, which the helper shows as `n/a`.

Responses are created with `store=False`, so the demo leaves no stored responses in the project.
Latency is printed too, but treat it as indicative only — it moves with load, not just caching.


In [7]:
RESULTS = []


def call(messages, label, demo="", **kwargs):
    """Send one Responses request and report the caching fields."""
    started = time.perf_counter()
    response = openai_client.responses.create(
        model=DEPLOYMENT,
        # The project endpoint requires an explicit item type on every input message.
        input=[{"type": "message", **message} for message in messages],
        max_output_tokens=32,
        store=False,
        **kwargs,
    )
    elapsed = time.perf_counter() - started

    usage = response.usage
    details = usage.input_tokens_details
    cached = (details.cached_tokens if details else 0) or 0
    written = getattr(details, "cache_write_tokens", None)

    record = {
        "demo": demo,
        "label": label,
        "input_tokens": usage.input_tokens,
        "cached_tokens": cached,
        "cache_write_tokens": written,
        "seconds": round(elapsed, 2),
    }
    RESULTS.append(record)

    written_text = "n/a" if written is None else str(written)
    share = (cached / usage.input_tokens * 100) if usage.input_tokens else 0
    print(
        f"{label:<34} "
        f"input={usage.input_tokens:>6}  "
        f"cached={cached:>6} ({share:5.1f}%)  "
        f"written={written_text:>6}  "
        f"{elapsed:5.2f}s"
    )
    return record


## Building a prefix that is actually cacheable

Two conditions have to hold together: **a minimum of 1,024 tokens in length**, and
**the first 1,024 tokens must be identical** across requests. Below that floor nothing caches,
no matter how stable your text is — which is the single most common reason a team sees
`cached_tokens` stuck at zero.

The cell below builds a stable instruction block and measures it, so you start from a prefix
that clears the floor with room to spare.


In [8]:
import tiktoken

POLICY_CLAUSES = [
    "Answer only from the supplied support policy. If the policy does not cover the question, "
    "say so plainly and route the customer to a human agent.",
    "Never quote an internal case identifier, an engineer's name, or an unpublished remediation date.",
    "Where a policy clause sets a time window, state the window and the clause it comes from.",
    "Treat entitlement questions and billing questions as separate: answer one, then offer the other.",
    "Do not speculate about root cause. Describe only what the policy says has been confirmed.",
    "When two clauses conflict, apply the more restrictive one and say which you applied.",
    "Use plain language. Expand every acronym on first use in the reply.",
    "Close every answer with the single next action the customer should take.",
]

# Repeat the clause set to clear the 1,024-token floor comfortably.
STABLE_PREFIX = "SUPPORT POLICY — OPERATING INSTRUCTIONS\n\n" + "\n".join(
    f"{i}. {clause}" for i, clause in enumerate(POLICY_CLAUSES * 12, start=1)
)

# Offline estimate; the service's own count appears as input_tokens in each call below.
tokens = len(tiktoken.get_encoding("o200k_base").encode(STABLE_PREFIX))
print(f"stable prefix: {tokens} tokens")
print("clears the 1,024-token minimum" if tokens >= 1024 else "TOO SHORT — increase the repeat count")


stable prefix: 1917 tokens
clears the 1,024-token minimum


---

## Demo 1 — the cold call writes, the warm call reads

The first request writes an eligible prefix to the cache. A later request with the same prefix
finds the matching entry and reuses the saved state instead of processing those tokens again.
It still processes the new input — and output tokens are always generated fresh.

**Watch for:** `cached=0` on the first call, then a large `cached` value on the second.
If the second call also reports zero, wait a moment and re-run — routing is not guaranteed.


In [9]:
def ask(question):
    return [
        {"role": "system", "content": STABLE_PREFIX},
        {"role": "user", "content": question},
    ]


call(ask("What should I do if the policy does not cover the question?"), "1. cold  — first sight", demo="1")
time.sleep(2)
call(ask("How should conflicting clauses be handled?"), "2. warm  — same prefix", demo="1")
time.sleep(2)
call(ask("What must every answer close with?"), "3. warm  — same prefix again", demo="1")


1. cold  — first sight             input=  1940  cached=  1937 ( 99.8%)  written=     0   4.18s
2. warm  — same prefix             input=  1934  cached=  1931 ( 99.8%)  written=     0   2.79s
3. warm  — same prefix again       input=  1934  cached=  1931 ( 99.8%)  written=     0   2.98s


{'demo': '1',
 'label': '3. warm  — same prefix again',
 'input_tokens': 1934,
 'cached_tokens': 1931,
 'cache_write_tokens': 0,
 'seconds': 2.98}

---

## Demo 2 — one character is enough to lose it

**A single character difference in the first 1,024 tokens results in a cache miss**, reported as
`cached_tokens = 0`. This is the demonstration worth running live in front of a customer,
because the failure is invisible in the response itself — the answer comes back perfectly normal.

Below, the second call adds one full stop to the very start of the prefix. Nothing else changes.


In [10]:
call(ask("Summarise the escalation rule."), "1. unchanged prefix", demo="2")
time.sleep(2)

mutated = [
    {"role": "system", "content": "." + STABLE_PREFIX},  # one character, at the front
    {"role": "user", "content": "Summarise the escalation rule."},
]
call(mutated, "2. one character added", demo="2")
time.sleep(2)

call(ask("Summarise the escalation rule."), "3. back to the original", demo="2")


1. unchanged prefix                input=  1934  cached=  1920 ( 99.3%)  written=    11   2.36s
2. one character added             input=  1934  cached=     0 (  0.0%)  written=  1931   2.45s
3. back to the original            input=  1934  cached=  1931 ( 99.8%)  written=     0   2.99s


{'demo': '2',
 'label': '3. back to the original',
 'input_tokens': 1934,
 'cached_tokens': 1931,
 'cache_write_tokens': 0,
 'seconds': 2.99}

---

## Demo 3 — static content first, dynamic content last

The same three facts, arranged two ways. The **dynamic-first** layout puts a timestamp and a
session identifier at the very front — exactly the pattern that quietly destroys cache hit rates
in production, because the prefix is different on every single call.

**Watch for:** the dynamic-first calls sitting at or near zero while the static-first calls reuse
most of the prefix.


In [ ]:
import uuid
from datetime import datetime, timezone


def dynamic_first(question):
    header = (
        f"Request time: {datetime.now(timezone.utc).isoformat()}\n"
        f"Session: {uuid.uuid4()}\n\n"
    )
    return [
        {"role": "system", "content": header + STABLE_PREFIX},
        {"role": "user", "content": question},
    ]


def static_first(question):
    footer = (
        f"\n\nRequest time: {datetime.now(timezone.utc).isoformat()}"
        f"\nSession: {uuid.uuid4()}"
    )
    return [
        {"role": "system", "content": STABLE_PREFIX},
        {"role": "user", "content": question + footer},
    ]


print("dynamic values at the FRONT of the prompt")
for n in range(1, 4):
    call(dynamic_first("State the plain-language rule."), f"  dynamic-first call {n}", demo="3-bad")
    time.sleep(1)

print()
print("same values moved to the END of the prompt")
for n in range(1, 4):
    call(static_first("State the plain-language rule."), f"  static-first call {n}", demo="3-good")
    time.sleep(1)


---

## Demo 4 — append-only conversations keep the cache

Keep conversation context **append-only**. Trimming, summarising or reordering earlier messages
changes the prefix and invalidates everything cached before them — so the cheap fix of "drop the
oldest turns to save tokens" often costs more than it saves.

**Watch for:** `cached` climbing turn by turn in the append-only run, then collapsing the moment
the trimmed run removes an early message.


In [ ]:
QUESTIONS = [
    "What is the first rule about coverage?",
    "And what about identifiers?",
    "How should time windows be stated?",
    "What happens when two clauses conflict?",
]

print("append-only — history grows at the end")
history = [{"role": "system", "content": STABLE_PREFIX}]
for turn, question in enumerate(QUESTIONS, start=1):
    history.append({"role": "user", "content": question})
    call(list(history), f"  turn {turn} (append-only)", demo="4-good")
    history.append({"role": "assistant", "content": f"Noted for turn {turn}."})
    time.sleep(1)

print()
print("trimmed — the oldest exchange is dropped each turn")
history = [{"role": "system", "content": STABLE_PREFIX}]
for turn, question in enumerate(QUESTIONS, start=1):
    history.append({"role": "user", "content": question})
    if turn > 2:
        # Drop the oldest exchange, keeping the system message. This rewrites the prefix.
        del history[1:3]
    call(list(history), f"  turn {turn} (trimmed)", demo="4-bad")
    history.append({"role": "assistant", "content": f"Noted for turn {turn}."})
    time.sleep(1)


---

## Demo 5 — explicit breakpoints and a cache key

> **Requires GPT-5.6 or later on a Standard pay-as-you-go deployment.**
> Earlier models return a 400 error for these parameters; PTU-M deployments do not support
> breakpoints. The cell catches that error and explains it rather than failing loudly.

Two controls are in play:

- **`prompt_cache_breakpoint`** marks the end of a reusable prefix. The breakpoint includes the
  block it sits on and everything before it. **Content after the breakpoint can change without
  invalidating the cached prefix** — which is the whole point.
- **`prompt_cache_key`** improves cache matching for requests that share a long common prefix.
  Reuse one key per prefix family. Past roughly **15 requests per minute** for the same prefix
  and key combination some requests might miss, so split across keys at volume while keeping a
  stable key-to-prefix mapping.

`prompt_cache_options.mode` is `implicit` by default — Azure OpenAI places a breakpoint on the
latest message and also uses any explicit breakpoints you provide. In `explicit` mode only your
breakpoints count, and a request with none does not use caching or incur cache-write charges.


In [ ]:
from openai import BadRequestError

CACHE_KEY = "demo:support-policy-v1"
CACHE_OPTIONS = {"mode": "explicit", "ttl": "30m"}

explicit_messages = [
    {
        "role": "system",
        "content": [
            {
                "type": "input_text",
                "text": STABLE_PREFIX,
                "prompt_cache_breakpoint": {"mode": "explicit"},
            }
        ],
    },
    {"role": "user", "content": "Give me the closing-action rule."},
]

try:
    call(
        explicit_messages,
        "1. explicit breakpoint (write)",
        demo="5",
        prompt_cache_key=CACHE_KEY,
        prompt_cache_options=CACHE_OPTIONS,
    )
    time.sleep(2)

    explicit_messages[1] = {"role": "user", "content": "Now give me the acronym rule instead."}
    call(
        explicit_messages,
        "2. same prefix, new question",
        demo="5",
        prompt_cache_key=CACHE_KEY,
        prompt_cache_options=CACHE_OPTIONS,
    )
except BadRequestError as error:
    print("Explicit breakpoints were rejected by this deployment.")
    print("Expected when the model is earlier than GPT-5.6, or the deployment is PTU-M.")
    print("Those models still cache automatically — demos 1 to 4 remain valid.")
    print(f"\nDetail: {error}")


---

## Demo 6 — the operational view

In production the useful signal is not a single call, it is the **ratio** of cache reads to total
input tokens over time, per deployment. A persistent zero means the prefix is not matching at all.
A sudden drop means a recent change moved something into the prefix that should not be there.

Where `cache_write_tokens` is reported, compare write volume against later reads: on GPT-5.6 and
later families **cache writes can incur charges in addition to discounted cache reads**, so a
prefix that churns can cost more than one left alone.


In [ ]:
def summarise(rows):
    groups = {}
    for row in rows:
        bucket = groups.setdefault(row["demo"], {"input": 0, "cached": 0, "calls": 0})
        bucket["input"] += row["input_tokens"]
        bucket["cached"] += row["cached_tokens"]
        bucket["calls"] += 1

    print(f"{'demo':<10}{'calls':>7}{'input':>10}{'cached':>10}{'reuse':>9}")
    print("-" * 46)
    for demo in sorted(groups):
        bucket = groups[demo]
        share = bucket["cached"] / bucket["input"] * 100 if bucket["input"] else 0
        print(
            f"{demo:<10}{bucket['calls']:>7}{bucket['input']:>10}"
            f"{bucket['cached']:>10}{share:>8.1f}%"
        )

    total_input = sum(b["input"] for b in groups.values())
    total_cached = sum(b["cached"] for b in groups.values())
    overall = total_cached / total_input * 100 if total_input else 0
    print("-" * 46)
    print(f"{'overall':<10}{len(rows):>7}{total_input:>10}{total_cached:>10}{overall:>8.1f}%")


summarise(RESULTS)


---

## What the run should have shown

| Demo | Expected shape | The point |
|---|---|---|
| 1 | zero, then high | The first call pays; later calls reuse |
| 2 | high, zero, high | One character at the front costs you the whole prefix |
| 3 | `3-bad` near zero, `3-good` high | Layout is the lever, not configuration |
| 4 | `4-good` climbing, `4-bad` collapsing | Append; never rewrite from the front |
| 5 | second call reuses the prefix | Content after a breakpoint is free to change |
| 6 | one reuse ratio per demo | The number worth alerting on |

### If everything reported zero

1. **Check the prefix length first.** Below 1,024 tokens nothing caches. The token count cell
   above tells you where you stand.
2. **Check the model.** In-memory prompt cache retention is supported on Azure OpenAI models
   GPT-4o or newer, for models with chat-completion, completion, responses or real-time operations.
3. **Check the gap between calls.** In-memory caches typically clear within 5 to 10 minutes of
   inactivity and are always removed within one hour of last use.
4. **Re-run.** Cached states are machine-local; a request reuses a prefix only when it reaches a
   machine holding a matching, unexpired entry.

### Carry back to the deck

- Prompt caching changes **latency and cost only** — it has no effect on the output content.
- It is **enabled by default** for supported models; the work is in how you structure the prompt.
- Cache reads are billed at a discount on input token pricing for Standard deployment types, and
  up to a 100% discount on input tokens for Provisioned deployment types. **Check the Azure OpenAI
  pricing page for current rates before quoting any figure.**

### Sources

- [Prompt caching with Azure OpenAI in Microsoft Foundry Models](https://learn.microsoft.com/en-us/azure/foundry/openai/how-to/prompt-caching)
- [Prompt caching — OpenAI API](https://developers.openai.com/api/docs/guides/prompt-caching)
- [Enable semantic caching for LLM APIs in Azure API Management](https://learn.microsoft.com/en-us/azure/api-management/azure-openai-enable-semantic-caching)
